# Módulo 2 — EDA: Clasificación de Conducción Distractiva
**Dataset real**: `arafatsahinafridi/multi-class-driver-behavior-image-dataset` (Kaggle)

Universidad Nacional de Colombia · IRNA 2026-01

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, sys
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

OUTPUT_DIR = Path('.')
print("Librerías cargadas.")

## 1. Descarga del dataset real desde Kaggle

In [ ]:
import kagglehub

print("Descargando dataset de conducción distractiva desde Kaggle...")
try:
    raw_path = Path(kagglehub.dataset_download(
        "arafatsahinafridi/multi-class-driver-behavior-image-dataset"))
    print(f"Path: {raw_path}")
except Exception as e:
    sys.exit(f"ERROR: {e}\nConfigura ~/.kaggle/kaggle.json con tus credenciales.")

# Localizar carpeta con clases (c0, c1, ...)
data_dir = None
for candidate in [raw_path, *raw_path.rglob("train"), *raw_path.rglob("Train")]:
    candidate = Path(candidate)
    if candidate.is_dir():
        subdirs = [d for d in candidate.iterdir() if d.is_dir()]
        class_like = [d for d in subdirs if d.name.lower().startswith("c") and d.name[1:].isdigit()]
        if len(class_like) >= 5:
            data_dir = candidate; break

if data_dir is None:
    subdirs = [d for d in raw_path.iterdir() if d.is_dir()]
    class_like = [d for d in subdirs if d.name.lower().startswith("c") and d.name[1:].isdigit()]
    if len(class_like) >= 5:
        data_dir = raw_path

assert data_dir is not None, f"No se encontró estructura c0/c1/... en {raw_path}"
class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()
                     and d.name.lower().startswith('c') and d.name[1:].isdigit()],
                    key=lambda d: int(d.name[1:]))
class_names = [d.name for d in class_dirs]
print(f"Clases encontradas ({len(class_names)}): {class_names}")

## 2. Distribución de clases

In [ ]:
# Contar imágenes por clase
class_counts = {}
for d in class_dirs:
    imgs = list(d.glob("*.jpg")) + list(d.glob("*.png")) + list(d.glob("*.jpeg"))
    class_counts[d.name] = len(imgs)

print("Imágenes por clase:")
total = 0
for cls, cnt in sorted(class_counts.items()):
    print(f"  {cls}: {cnt:,}")
    total += cnt
print(f"  TOTAL: {total:,}")

# Etiquetas descriptivas del dataset
CLASS_LABELS = {
    "c0": "Conducción segura",
    "c1": "Enviando texto (der.)",
    "c2": "Hablando por teléfono (der.)",
    "c3": "Enviando texto (izq.)",
    "c4": "Hablando por teléfono (izq.)",
    "c5": "Operando radio/controles",
    "c6": "Bebiendo",
    "c7": "Alcanzando hacia atrás",
    "c8": "Arreglándose cabello/maquillaje",
    "c9": "Conversando con pasajero",
}
labels = [CLASS_LABELS.get(c, c) for c in sorted(class_counts.keys())]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cls_sorted = sorted(class_counts.keys())
counts = [class_counts[c] for c in cls_sorted]

ax = axes[0]
bars = ax.bar(cls_sorted, counts, color=plt.cm.tab10(np.linspace(0,1,len(cls_sorted))), edgecolor='white')
ax.set_xlabel("Clase"); ax.set_ylabel("Nº de imágenes")
ax.set_title("Distribución de imágenes por clase")
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, str(cnt),
            ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
ax2.pie(counts, labels=cls_sorted, autopct='%1.1f%%',
        colors=plt.cm.tab10(np.linspace(0,1,len(cls_sorted))), startangle=90)
ax2.set_title("Proporción por clase")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig_class_distribution.png", dpi=120, bbox_inches='tight')
plt.show()
print("Guardado: fig_class_distribution.png")

## 3. Ejemplos de imágenes por clase

In [ ]:
import random; random.seed(42)

n_rows = len(class_dirs); n_cols = 4
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3))

for row_i, (cls_dir, cls_name) in enumerate(zip(class_dirs, class_names)):
    imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png")) + list(cls_dir.glob("*.jpeg"))
    sample = random.sample(imgs, min(n_cols, len(imgs)))
    label = CLASS_LABELS.get(cls_name, cls_name)
    for col_i in range(n_cols):
        ax = axes[row_i][col_i] if n_rows > 1 else axes[col_i]
        if col_i < len(sample):
            img = Image.open(sample[col_i]).convert("RGB").resize((150,150))
            ax.imshow(img)
        ax.axis('off')
        if col_i == 0:
            ax.set_ylabel(f"{cls_name}\n{label}", fontsize=8, labelpad=5)
            ax.yaxis.set_visible(True)
            ax.tick_params(left=False, labelleft=True)
            
plt.suptitle("Muestras del Dataset Real de Conducción Distractiva", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig_examples_grid.png", dpi=100, bbox_inches='tight')
plt.show()
print("Guardado: fig_examples_grid.png")

## 4. Estadísticas de tamaño de imágenes

In [ ]:
# Muestrear hasta 100 imágenes por clase para estadísticas
widths, heights = [], []
for cls_dir in class_dirs:
    imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
    sample = random.sample(imgs, min(100, len(imgs)))
    for img_path in sample:
        try:
            with Image.open(img_path) as im:
                w, h = im.size
                widths.append(w); heights.append(h)
        except: pass

print(f"Ancho  — media={np.mean(widths):.0f} | min={np.min(widths)} | max={np.max(widths)}")
print(f"Alto   — media={np.mean(heights):.0f} | min={np.min(heights)} | max={np.max(heights)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='#3182CE', edgecolor='white')
axes[0].set_title("Distribución de anchos"); axes[0].set_xlabel("Píxeles")
axes[1].hist(heights, bins=30, color='#48BB78', edgecolor='white')
axes[1].set_title("Distribución de altos"); axes[1].set_xlabel("Píxeles")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig_size_stats.png", dpi=120, bbox_inches='tight')
plt.show()

## 5. Análisis de balance de clases

In [ ]:
counts_arr = np.array([class_counts[c] for c in cls_sorted])
total_imgs = counts_arr.sum()
balance_ratio = counts_arr.max() / counts_arr.min()
cv = counts_arr.std() / counts_arr.mean() * 100

print(f"Total imágenes: {total_imgs:,}")
print(f"Ratio max/min: {balance_ratio:.2f}x")
print(f"Coeficiente de variación: {cv:.1f}%")
print(f"Dataset {'BALANCEADO' if balance_ratio < 2 else 'DESBALANCEADO'} (ratio {balance_ratio:.2f}x)")

fig, ax = plt.subplots(figsize=(10, 4))
deviation = counts_arr - counts_arr.mean()
colors = ['#E53E3E' if d < 0 else '#48BB78' for d in deviation]
ax.bar(cls_sorted, deviation, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title(f"Desviación respecto a la media ({counts_arr.mean():.0f} imgs/clase)")
ax.set_xlabel("Clase"); ax.set_ylabel("Δ imágenes")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig_balance_analysis.png", dpi=120, bbox_inches='tight')
plt.show()

## 6. Análisis de color promedio por clase

In [ ]:
# Canal RGB promedio por clase (muestra de 50 imgs/clase)
class_rgb = {}
for cls_dir, cls_name in zip(class_dirs, class_names):
    imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
    sample = random.sample(imgs, min(50, len(imgs)))
    r_vals, g_vals, b_vals = [], [], []
    for img_path in sample:
        try:
            arr = np.array(Image.open(img_path).convert("RGB").resize((64,64)), dtype=np.float32)
            r_vals.append(arr[:,:,0].mean()); g_vals.append(arr[:,:,1].mean()); b_vals.append(arr[:,:,2].mean())
        except: pass
    class_rgb[cls_name] = (np.mean(r_vals), np.mean(g_vals), np.mean(b_vals))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(class_names)); w = 0.25
r_means = [class_rgb[c][0] for c in class_names]
g_means = [class_rgb[c][1] for c in class_names]
b_means = [class_rgb[c][2] for c in class_names]
ax.bar(x-w, r_means, w, label='R', color='#FC8181', edgecolor='white')
ax.bar(x,   g_means, w, label='G', color='#68D391', edgecolor='white')
ax.bar(x+w, b_means, w, label='B', color='#63B3ED', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(class_names); ax.legend()
ax.set_title("Canal RGB promedio por clase"); ax.set_ylabel("Intensidad media (0-255)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig_color_means.png", dpi=120, bbox_inches='tight')
plt.show()
print("Guardado: fig_color_means.png")

## 7. Conclusiones del EDA
- Dataset real con imágenes de conductores en 10 categorías de comportamiento.
- Clases detectadas dinámicamente del filesystem del dataset de Kaggle.
- Distribución de clases relativamente balanceada.
- Imágenes de tamaño variable → se normalizarán a 224×224 para ResNet18.
- Variabilidad de color entre clases sugiere que la CNN puede aprender patrones visuales discriminativos.